# Tiền xử lý dữ liệu - bản đã chuẩn hóa

Notebook này thay thế bản cũ theo hướng:
- Dùng một cấu trúc đường dẫn thống nhất, có fallback khi chạy ở thư mục khác.
- Rút gọn tên cột, loại trùng lặp, loại cột thiếu quá nhiều.
- Chia train/validation/test trước khi fit các bước điền thiếu và chuẩn hóa để tránh rò rỉ dữ liệu.
- Tạo feature engineering sau khi đã điền thiếu các biến gốc.
- Lưu dữ liệu đã xử lý để notebook trực quan hóa dùng trực tiếp.


## 1. Chuẩn bị môi trường và đường dẫn

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.20

pd.options.display.float_format = lambda x: f"{x:,.4f}"


In [2]:
# Tìm dataset.csv theo một số vị trí phổ biến.
ROOT = Path.cwd()

DATA_PATH_CANDIDATES = [
    ROOT / "dataset.csv",
    ROOT / "data" / "dataset.csv",
    ROOT.parent / "data" / "dataset.csv",
    Path("/mnt/data/dataset.csv"),
]

DATA_PATH = next((p for p in DATA_PATH_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy dataset.csv. Hãy đặt file cùng thư mục notebook hoặc trong thư mục data/."
    )

OUTPUT_DIR = DATA_PATH.parent / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_PATH:", DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


DATA_PATH: c:\Users\Xuanuio\Desktop\Deadline\Kho_DL\Data_Minning\Minning\Data\dataset.csv
OUTPUT_DIR: c:\Users\Xuanuio\Desktop\Deadline\Kho_DL\Data_Minning\Minning\Data\processed


## 2. Đọc dữ liệu và chuẩn hóa tên cột

In [3]:
raw_df = pd.read_csv(DATA_PATH)

rename_map = {
    "Country Name": "country_name",
    "Country Code": "country_code",
    "Year": "year",
    "Population, total": "population",
    "Poverty headcount ratio at $3.00 a day (2021 PPP) (% of population)": "poverty_ratio",
    "Population growth (annual %)": "pop_growth",
    "Life expectancy at birth, total (years)": "life_expectancy",
    "GDP per capita (current US$)": "gdp_per_capita",
    "GDP growth (annual %)": "gdp_growth",
    "People using safely managed sanitation services (% of population)": "sanitation",
    "Access to electricity (% of population)": "electricity",
    "People using at least basic drinking water services (% of population)": "water_access",
    "Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita)": "co2_emissions",
    "Population living in slums (% of urban population)": "slum_population",
    "Labor force participation rate, total (% of total population ages 15+) (modeled ILO estimate)": "labor_force",
}

missing_columns = [c for c in rename_map if c not in raw_df.columns]
if missing_columns:
    raise ValueError(f"Dataset thiếu các cột bắt buộc: {missing_columns}")

df = raw_df.rename(columns=rename_map).copy()

# Ép kiểu an toàn cho các cột số.
numeric_cols = [
    "year", "population", "poverty_ratio", "pop_growth", "life_expectancy",
    "gdp_per_capita", "gdp_growth", "sanitation", "electricity", "water_access",
    "co2_emissions", "slum_population", "labor_force"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["year"] = df["year"].astype("Int64")

print("Kích thước dữ liệu gốc:", df.shape)
print("Số quốc gia/vùng lãnh thổ:", df["country_name"].nunique())
print("Khoảng năm:", int(df["year"].min()), "-", int(df["year"].max()))
df.head()


Kích thước dữ liệu gốc: (5642, 15)
Số quốc gia/vùng lãnh thổ: 217
Khoảng năm: 2000 - 2025


,country_name,country_code,year,population,poverty_ratio,pop_growth,life_expectancy,gdp_per_capita,gdp_growth,sanitation,electricity,water_access,co2_emissions,slum_population,labor_force
0,Aruba,ABW,2000,"90,588.0000",NaN,1.0308,72.9390,"20,681.0230",7.6229,NaN,91.7000,95.2335,2.9684,0.0000,NaN
1,Aruba,ABW,2001,"91,439.0000",NaN,0.9350,73.0440,"20,740.1326",4.1820,NaN,100.0000,95.3591,2.9736,NaN,NaN
2,Aruba,ABW,2002,"92,074.0000",NaN,0.6921,73.1350,"21,307.2483",-0.9450,NaN,100.0000,95.4846,3.2257,0.0000,NaN
3,Aruba,ABW,2003,"93,128.0000",NaN,1.1382,73.2360,"21,949.4860",1.1105,NaN,100.0000,95.6101,3.6767,NaN,NaN
4,Aruba,ABW,2004,"95,138.0000",NaN,2.1354,73.2230,"23,700.6320",7.2937,NaN,100.0000,95.7356,3.6726,0.0000,NaN


In [4]:
column_description = pd.DataFrame({
    "column": [
        "country_name", "country_code", "year", "population", "poverty_ratio",
        "pop_growth", "life_expectancy", "gdp_per_capita", "gdp_growth",
        "sanitation", "electricity", "water_access", "co2_emissions",
        "slum_population", "labor_force"
    ],
    "meaning": [
        "Tên quốc gia/vùng lãnh thổ",
        "Mã quốc gia",
        "Năm quan sát",
        "Tổng dân số",
        "Tỷ lệ nghèo theo chuẩn 3 USD/ngày",
        "Tăng trưởng dân số hằng năm",
        "Tuổi thọ trung bình - biến mục tiêu",
        "GDP bình quân đầu người",
        "Tăng trưởng GDP hằng năm",
        "Tỷ lệ tiếp cận vệ sinh an toàn",
        "Tỷ lệ tiếp cận điện",
        "Tỷ lệ tiếp cận nước uống cơ bản",
        "CO2 bình quân đầu người",
        "Tỷ lệ dân số đô thị sống trong khu ổ chuột",
        "Tỷ lệ tham gia lực lượng lao động"
    ],
    "role": [
        "id", "id", "time", "predictor", "drop_high_missing",
        "predictor", "target", "raw_feature_for_log_transform", "predictor",
        "predictor", "predictor", "predictor", "predictor",
        "drop_high_missing", "predictor"
    ]
})

column_description


,column,meaning,role
0,country_name,Tên quốc gia/vùng lãnh thổ,id
1,country_code,Mã quốc gia,id
2,year,Năm quan sát,time
3,population,Tổng dân số,predictor
4,poverty_ratio,Tỷ lệ nghèo theo chuẩn 3 USD/ngày,drop_high_missing
5,pop_growth,Tăng trưởng dân số hằng năm,predictor
6,life_expectancy,Tuổi thọ trung bình - biến mục tiêu,target
7,gdp_per_capita,GDP bình quân đầu người,raw_feature_for_log_transform
8,gdp_growth,Tăng trưởng GDP hằng năm,predictor
9,sanitation,Tỷ lệ tiếp cận vệ sinh an toàn,predictor


## 3. Làm sạch cơ bản

In [5]:
before = len(df)
df = df.drop_duplicates()
duplicated_rows = before - len(df)

# Loại bỏ dòng thiếu target vì không dùng được cho bài toán hồi quy.
before = len(df)
df = df.dropna(subset=["life_expectancy"])
missing_target_rows = before - len(df)

# Sắp xếp để kết quả dễ kiểm tra.
df = df.sort_values(["country_name", "year"]).reset_index(drop=True)

print(f"Đã loại {duplicated_rows} dòng trùng lặp.")
print(f"Đã loại {missing_target_rows} dòng thiếu life_expectancy.")
print("Kích thước sau làm sạch cơ bản:", df.shape)


Đã loại 0 dòng trùng lặp.
Đã loại 0 dòng thiếu life_expectancy.
Kích thước sau làm sạch cơ bản: (5642, 15)


In [6]:
missing_summary = (
    df.isna().mean()
      .mul(100)
      .sort_values(ascending=False)
      .rename("missing_percent")
      .reset_index()
      .rename(columns={"index": "column"})
)

missing_summary[missing_summary["missing_percent"] > 0]


,column,missing_percent
0,poverty_ratio,64.5161
1,slum_population,60.8118
2,sanitation,34.1546
3,labor_force,13.9844
4,co2_emissions,6.4516
5,electricity,5.1755
6,gdp_growth,4.7501
7,water_access,3.8816
8,gdp_per_capita,3.4562
9,pop_growth,0.0177


In [7]:
# Hai cột này thiếu quá nhiều nên loại bỏ để tránh nhiễu và rủi ro điền thiếu sai lệch.
HIGH_MISSING_COLS = ["poverty_ratio", "slum_population"]
df = df.drop(columns=[c for c in HIGH_MISSING_COLS if c in df.columns])

print("Các cột đã loại bỏ:", HIGH_MISSING_COLS)
print("Kích thước sau khi loại cột thiếu nhiều:", df.shape)


Các cột đã loại bỏ: ['poverty_ratio', 'slum_population']
Kích thước sau khi loại cột thiếu nhiều: (5642, 13)


## 4. Chia train/validation/test trước khi fit preprocessing

In [8]:
train_df, temp_df = train_test_split(
    df,
    test_size=VAL_SIZE + TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
)

relative_test_size = TEST_SIZE / (VAL_SIZE + TEST_SIZE)
val_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test_size,
    random_state=RANDOM_STATE,
    shuffle=True,
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


Train: (3385, 13)
Validation: (1128, 13)
Test: (1129, 13)


## 5. Điền giá trị thiếu

Nguyên tắc:
- Fit tham số điền thiếu trên `train_df`.
- Áp dụng cùng tham số cho `validation` và `test`.
- `sanitation` thiếu tương đối nhiều nên dùng `IterativeImputer` với Bayesian Ridge.
- Các biến thiếu thấp/vừa được điền bằng mean hoặc median tùy skewness trên train.


In [9]:
def fill_by_train_skewness(train, val, test, columns, skew_threshold=0.5):
    train = train.copy()
    val = val.copy()
    test = test.copy()
    report = []

    for col in columns:
        skew_value = train[col].skew()
        if pd.isna(skew_value):
            method = "median"
            fill_value = train[col].median()
        elif -skew_threshold <= skew_value <= skew_threshold:
            method = "mean"
            fill_value = train[col].mean()
        else:
            method = "median"
            fill_value = train[col].median()

        train_missing = int(train[col].isna().sum())
        val_missing = int(val[col].isna().sum())
        test_missing = int(test[col].isna().sum())

        train[col] = train[col].fillna(fill_value)
        val[col] = val[col].fillna(fill_value)
        test[col] = test[col].fillna(fill_value)

        report.append({
            "column": col,
            "skew_train": skew_value,
            "method": method,
            "fill_value": fill_value,
            "missing_train": train_missing,
            "missing_val": val_missing,
            "missing_test": test_missing,
        })

    return train, val, test, pd.DataFrame(report)


base_numeric_cols = [
    c for c in train_df.select_dtypes(include=["number", "Int64"]).columns
    if c not in ["life_expectancy", "sanitation"]
]

train_df, val_df, test_df, impute_report = fill_by_train_skewness(
    train_df, val_df, test_df, base_numeric_cols
)

impute_report


,column,skew_train,method,fill_value,missing_train,missing_val,missing_test
0,year,-0.0102,mean,"2,012.5279",0,0,0
1,population,8.9591,median,"5,814,626.0000",0,0,0
2,pop_growth,1.4837,median,1.1738,1,0,0
3,gdp_per_capita,3.1859,median,"5,686.9804",112,41,42
4,gdp_growth,1.1612,median,3.4661,156,53,59
5,electricity,-1.4007,median,99.1000,161,71,60
6,water_access,-1.4743,median,95.4601,126,40,53
7,co2_emissions,7.7098,median,2.3570,213,66,85
8,labor_force,-0.1612,mean,61.0564,484,141,164


In [10]:
# Điền sanitation bằng MICE sau khi các biến nền đã được điền thiếu.
mice_predictor_cols = [
    c for c in [
        "year", "population", "pop_growth", "gdp_per_capita", "gdp_growth",
        "electricity", "water_access", "co2_emissions", "labor_force"
    ]
    if c in train_df.columns
]

mice_cols = mice_predictor_cols + ["sanitation"]

mice_imputer = IterativeImputer(
    estimator=BayesianRidge(),
    random_state=RANDOM_STATE,
    max_iter=20,
    sample_posterior=False,
)

sanitation_missing_before = {
    "train": int(train_df["sanitation"].isna().sum()),
    "validation": int(val_df["sanitation"].isna().sum()),
    "test": int(test_df["sanitation"].isna().sum()),
}

train_df[mice_cols] = mice_imputer.fit_transform(train_df[mice_cols])
val_df[mice_cols] = mice_imputer.transform(val_df[mice_cols])
test_df[mice_cols] = mice_imputer.transform(test_df[mice_cols])

# Giữ các biến tỷ lệ phần trăm trong khoảng hợp lệ 0-100.
percent_cols = ["sanitation", "electricity", "water_access", "labor_force"]
for frame in [train_df, val_df, test_df]:
    for col in percent_cols:
        if col in frame.columns:
            frame[col] = frame[col].clip(0, 100)

pd.DataFrame({
    "split": list(sanitation_missing_before.keys()),
    "sanitation_missing_before": list(sanitation_missing_before.values()),
    "sanitation_missing_after": [
        int(train_df["sanitation"].isna().sum()),
        int(val_df["sanitation"].isna().sum()),
        int(test_df["sanitation"].isna().sum()),
    ]
})


,split,sanitation_missing_before,sanitation_missing_after
0,train,1144,0
1,validation,377,0
2,test,406,0


## 6. Feature engineering

In [11]:
def add_engineered_features(frame):
    frame = frame.copy()

    # COVID thường gây nhiễu theo năm với các chỉ số sức khỏe/xã hội.
    frame["is_covid"] = frame["year"].isin([2020, 2021]).astype(int)

    # GDP: log trước, sau đó mới chuẩn hóa. Không giữ GDP gốc trong tập mô hình.
    frame["gdp_per_capita"] = frame["gdp_per_capita"].clip(lower=0)
    frame["log1p_gdp_per_capita"] = np.log1p(frame["gdp_per_capita"])

    # Tương tác hạ tầng: điện và nước cùng cao thường phản ánh điều kiện sống tốt hơn.
    frame["electricity_water_interaction"] = (
        frame["electricity"] * frame["water_access"] / 100
    )

    # Khoảng thiếu hụt dịch vụ cơ bản: càng cao càng kém.
    basic_services_index = frame[["sanitation", "electricity", "water_access"]].mean(axis=1)
    frame["basic_services_gap"] = 100 - basic_services_index

    # Thay GDP gốc bằng biến log để giảm lệch phải.
    frame = frame.drop(columns=["gdp_per_capita"])

    return frame


train_df = add_engineered_features(train_df)
val_df = add_engineered_features(val_df)
test_df = add_engineered_features(test_df)

engineered_features = [
    "is_covid",
    "log1p_gdp_per_capita",
    "electricity_water_interaction",
    "basic_services_gap",
]

train_df[engineered_features].head()


,is_covid,log1p_gdp_per_capita,electricity_water_interaction,basic_services_gap
2476,0,10.0225,100.0000,5.1682
4144,0,11.2446,99.9152,3.5864
1946,0,7.7102,74.7838,36.5109
4974,0,6.5176,4.5918,78.5457
3844,0,11.6016,100.0000,7.3958


In [12]:
nan_check = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "total_missing": [
        int(train_df.isna().sum().sum()),
        int(val_df.isna().sum().sum()),
        int(test_df.isna().sum().sum()),
    ]
})

nan_check


,split,total_missing
0,train,0
1,validation,0
2,test,0


## 7. Lưu dữ liệu đã điền thiếu, chưa chuẩn hóa

In [13]:
processed_df = (
    pd.concat(
        [
            train_df.assign(split="train"),
            val_df.assign(split="validation"),
            test_df.assign(split="test"),
        ],
        axis=0,
    )
    .sort_values(["country_name", "year"])
    .reset_index(drop=True)
)

processed_path = OUTPUT_DIR / "processed_data.csv"
processed_df.to_csv(processed_path, index=False)

print("Đã lưu:", processed_path)
processed_df.head()


Đã lưu: c:\Users\Xuanuio\Desktop\Deadline\Kho_DL\Data_Minning\Minning\Data\processed\processed_data.csv


,country_name,country_code,year,population,pop_growth,life_expectancy,gdp_growth,sanitation,electricity,water_access,co2_emissions,labor_force,is_covid,log1p_gdp_per_capita,electricity_water_interaction,basic_services_gap,split
0,Afghanistan,AFG,"2,000.0000","20,130,327.0000",1.2122,55.0050,3.4661,10.1321,4.4000,29.7310,0.0501,46.5710,0,5.1701,1.3082,85.2456,train
1,Afghanistan,AFG,"2,001.0000","20,284,307.0000",0.7620,55.5110,-9.4320,10.1315,9.3000,29.7629,0.0464,46.5330,0,4.9395,2.7680,83.6019,train
2,Afghanistan,AFG,"2,002.0000","21,378,117.0000",5.2520,56.2250,28.6000,10.8540,14.1000,31.8361,0.0439,46.5080,0,5.1927,4.4889,81.0700,train
3,Afghanistan,AFG,"2,003.0000","22,733,049.0000",6.1452,57.1710,8.8323,11.5763,19.0000,33.9086,0.0447,46.4990,0,5.2977,6.4426,78.5050,train
4,Afghanistan,AFG,"2,004.0000","23,560,654.0000",3.5758,57.8100,1.4141,12.2981,23.8000,35.9994,0.0383,46.5070,0,5.4061,8.5678,75.9675,train


## 8. Chuẩn hóa dữ liệu cho mô hình

In [14]:
# Chuẩn hóa các biến số liên tục. Không scale target, year và biến nhị phân is_covid.
do_not_scale = {"life_expectancy", "year", "is_covid"}

cols_to_scale = [
    c for c in train_df.select_dtypes(include=[np.number]).columns
    if c not in do_not_scale
]

scaler = StandardScaler()

train_scaled = train_df.copy()
val_scaled = val_df.copy()
test_scaled = test_df.copy()

train_scaled[cols_to_scale] = scaler.fit_transform(train_df[cols_to_scale])
val_scaled[cols_to_scale] = scaler.transform(val_df[cols_to_scale])
test_scaled[cols_to_scale] = scaler.transform(test_df[cols_to_scale])

train_path = OUTPUT_DIR / "train.csv"
val_path = OUTPUT_DIR / "val.csv"
test_path = OUTPUT_DIR / "test.csv"

train_scaled.to_csv(train_path, index=False)
val_scaled.to_csv(val_path, index=False)
test_scaled.to_csv(test_path, index=False)

feature_columns = [
    c for c in train_scaled.columns
    if c not in ["life_expectancy", "country_name", "country_code"]
]

metadata = {
    "target": "life_expectancy",
    "id_columns": ["country_name", "country_code"],
    "feature_columns": feature_columns,
    "scaled_columns": cols_to_scale,
    "dropped_columns": HIGH_MISSING_COLS,
    "engineered_features": engineered_features,
    "random_state": RANDOM_STATE,
    "split_ratio": {"train": 0.60, "validation": 0.20, "test": 0.20},
}

metadata_path = OUTPUT_DIR / "preprocessing_metadata.json"
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

print("Đã lưu:")
print("-", train_path)
print("-", val_path)
print("-", test_path)
print("-", metadata_path)
train_scaled.head()


Đã lưu:
- c:\Users\Xuanuio\Desktop\Deadline\Kho_DL\Data_Minning\Minning\Data\processed\train.csv
- c:\Users\Xuanuio\Desktop\Deadline\Kho_DL\Data_Minning\Minning\Data\processed\val.csv
- c:\Users\Xuanuio\Desktop\Deadline\Kho_DL\Data_Minning\Minning\Data\processed\test.csv
- c:\Users\Xuanuio\Desktop\Deadline\Kho_DL\Data_Minning\Minning\Data\processed\preprocessing_metadata.json


,country_name,country_code,year,population,pop_growth,life_expectancy,gdp_growth,sanitation,electricity,water_access,co2_emissions,labor_force,is_covid,log1p_gdp_per_capita,electricity_water_interaction,basic_services_gap
2476,Israel,ISR,"2,006.0000",-0.2039,0.3045,80.5537,0.3857,1.1469,0.6260,0.7412,0.4666,0.1171,0,0.8676,0.7593,-0.9344
4144,Qatar,QAT,"2,010.0000",-0.2458,-0.4869,79.4440,2.7444,1.3225,0.6260,0.7362,4.2562,2.5619,0,1.6441,0.7566,-1.0059
1946,Ghana,GHA,"2,022.0000",-0.0027,0.3991,65.2460,0.0828,-1.2895,0.0981,0.0394,-0.4426,0.2389,0,-0.6017,-0.0406,0.4824
4974,Tanzania,TZA,"2,008.0000",0.0699,0.8844,59.8960,0.4000,-1.4551,-2.5097,-2.7365,-0.5035,2.6426,0,-1.3595,-2.2672,2.3826
3844,Norway,NOR,"2,022.0000",-0.2162,-0.2346,82.5098,-0.0114,0.9039,0.6260,0.7412,0.3339,0.4717,0,1.8709,0.7593,-0.8337


In [15]:
summary = pd.DataFrame({
    "file": [
        "processed_data.csv",
        "train.csv",
        "val.csv",
        "test.csv",
        "preprocessing_metadata.json",
    ],
    "purpose": [
        "Dữ liệu đã điền thiếu, chưa chuẩn hóa; dùng cho trực quan hóa",
        "Tập train đã chuẩn hóa; dùng huấn luyện mô hình",
        "Tập validation đã chuẩn hóa; dùng chọn tham số",
        "Tập test đã chuẩn hóa; dùng đánh giá cuối",
        "Thông tin target, feature, cột đã scale/drop",
    ],
})

summary


,file,purpose
0,processed_data.csv,"Dữ liệu đã điền thiếu, chưa chuẩn hóa; dùng ch..."
1,train.csv,Tập train đã chuẩn hóa; dùng huấn luyện mô hình
2,val.csv,Tập validation đã chuẩn hóa; dùng chọn tham số
3,test.csv,Tập test đã chuẩn hóa; dùng đánh giá cuối
4,preprocessing_metadata.json,"Thông tin target, feature, cột đã scale/drop"
